In [1]:
%matplotlib agg

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import random

from src.model import (
    load_data, explore_data, split_data, EDA, data_cleaning,
    encode_data, select_model, compare_ensembles,
    tune_hyperparameters, important_features, evaluate_model
)

In [2]:
seed = random.randint(1000, 9999)
print("Random seed: ", seed)

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "term-deposit-marketing-2020.csv"

Random seed:  5639


In [3]:
term_deposit_df = load_data(DATA_PATH)
numeric_df, categorical_df = explore_data(term_deposit_df)

       age           job   marital  education default  balance housing loan  \
0       58    management   married   tertiary      no     2143     yes   no   
1       44    technician    single  secondary      no       29     yes   no   
2       33  entrepreneur   married  secondary      no        2     yes  yes   
3       47   blue-collar   married    unknown      no     1506     yes   no   
4       33       unknown    single    unknown      no        1      no   no   
39995   53    technician   married   tertiary      no      395      no   no   
39996   30    management    single   tertiary      no     3340      no   no   
39997   54         admin  divorced  secondary      no      200      no   no   
39998   34    management   married   tertiary      no     1047      no   no   
39999   38    technician   married  secondary      no     1442     yes   no   

        contact  day month  duration  campaign    y  
0       unknown    5   may       261         1   no  
1       unknown    5  

In [4]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    term_deposit_df, target="y", seed=seed
)

Size of the training set: 24000
Size of the validation set: 12800
Size of the testing set: 3200


In [5]:
df_tv = EDA(X_train, y_train, X_val, y_val, categorical_df, numeric_df)


Categorical Value Counts (Train + Val)

job
blue-collar      8617
management       7535
technician       6296
admin            4134
services         3601
retired          1327
self-employed    1297
entrepreneur     1284
unemployed       1024
housemaid         991
student           479
unknown           215
 
marital
married     22438
single      10031
divorced     4331
 
education
secondary    19333
tertiary     10335
primary       5740
unknown       1392
 
default
no     36064
yes      736
 
housing
yes    22131
no     14669
 
loan
no     30441
yes     6359
 
contact
cellular     22944
unknown      11725
telephone     2131
 
month
may    12427
jul     5892
aug     4802
jun     4336
nov     3309
apr     2504
feb     2117
jan     1087
mar      238
oct       76
dec       12
 
y
no     34136
yes     2664
 


In [6]:
term_deposit_df, cat_mode, num_bounds = data_cleaning(
    df_tv, categorical_df, numeric_df
)

In [7]:
X_train, X_val, y_train, y_val, le_dict, scaler, le_y = encode_data(
    X_train, X_val, y_train, y_val, categorical_df, numeric_df
)

In [8]:
models, predictions = select_model(X_train, X_val, y_train, y_val)
print(models)

  0%|          | 0/31 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 1738, number of negative: 22262
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000712 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 692
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.072417 -> initscore=-2.550146
[LightGBM] [Info] Start training from score -2.550146

Best model for minority class: NearestCentroid
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  \
Model                                                                           
NearestCentroid                    0.89               0.80     0.80      0.90   
XGBClassifier                      0.94               0.71     0.71      0.93   
DecisionTreeClassifier             0.91               0.70     0.70     

In [9]:
fitted_models, results_df = compare_ensembles(
    X_train, y_train, X_val, y_val, seed, cv=5
)
print(results_df)

           Model  Minority_Recall
2           LGBM             0.89
3         Voting             0.52
4       Stacking             0.49
0            KNN             0.26
1  Random Forest             0.23


In [10]:
best_model, best_params, best_score = tune_hyperparameters(
    X_train, y_train, X_val, y_val, seed
)

100%|██████████| 20/20 [00:07<00:00,  2.65trial/s, best loss: -0.9072822822822822]
Best Parameters: {'n_estimators': 125, 'max_depth': 5, 'learning_rate': np.float64(0.15405869043525727), 'num_leaves': 10, 'min_child_samples': 25, 'subsample': np.float64(0.7116992962923631), 'colsample_bytree': np.float64(0.8206442130870457)}
Best Score: 0.91


In [11]:
feature_df = important_features(X_train, best_model)
print(feature_df)

      Feature  Importance
0       month         257
1    duration         254
2         day         154
3     balance         128
4         age          87
5     contact          54
6         job          47
7    campaign          46
8   education          31
9     marital          26
10    housing          21
11       loan          12
12    default           8


In [12]:
clf_report = evaluate_model(
    best_model, X_test, y_test,
    categorical_df, numeric_df,
    cat_mode, num_bounds, le_dict, scaler, le_y
)
print(clf_report)

              precision    recall  f1-score   support

          no       0.99      0.77      0.87      2968
         yes       0.24      0.92      0.38       232

    accuracy                           0.78      3200
   macro avg       0.62      0.85      0.62      3200
weighted avg       0.94      0.78      0.83      3200

